# Fetch NMDC taxonomy summary files (issue #77)

Downloads the NOW-tier taxonomy summary files from `data.microbiomedata.org` via HTTP,
parses each format, and writes one Parquet file per `data_object_type`.

Resolves issue #57 for small files: direct HTTP download is sufficient (no Globus needed).

| `data_object_type` | ~files | ~total | Format |
|---|---|---|---|
| `GOTTCHA2 Classification Report` | 4,243 | 0.1 GB | headered TSV |
| `Kraken2 Classification Report` | 4,243 | 2.1 GB | 6-col no-header TSV |
| `Centrifuge output report file` | — | — | headered TSV |
| `GTDBTK Bacterial Summary` | 3,535 | ~0 GB | headered TSV, one row per MAG bin |
| `GTDBTK Archaeal Summary` | 3,535 | ~0 GB | headered TSV, one row per MAG bin |
| `CheckM Statistics` | 3,535 | ~0 GB | fixed-width (custom parser) |
| `Annotation Statistics` | 4,757 | ~0 GB | tab-sep multi-section: sequences, genes, quality (one wide row per run) |

Each output Parquet includes a `workflow_run_id` column (= `was_generated_by` from `data_object_set`).
Biosample join requires a subsequent join via `workflow_execution_set.was_informed_by`.

In [1]:
import inspect
import io
import logging
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential

# get_spark_session is auto-imported by the BERDL kernel startup script
# (~/.ipython/profile_default/startup/00-notebookutils.py).

OUT_DIR = Path(os.environ.get("TAXONOMY_OUT_DIR", "loaded_taxonomy"))
OUT_DIR.mkdir(exist_ok=True)

# On-disk cache of raw downloaded files. Re-runs (e.g. after a parser tweak) skip the
# HTTP fetch entirely. Delete the directory to force a fresh download.
CACHE_DIR = Path(os.environ.get("DOWNLOAD_CACHE_DIR", str(OUT_DIR / "raw_cache")))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Parallel HTTP workers for downloading data files.
HTTP_WORKERS = int(os.environ.get("HTTP_WORKERS", "32"))

# Set to True to stop after the first file of each type (for testing).
DRY_RUN = False

# ── Logging: each run gets its own timestamped log file under OUT_DIR/logs/ ──
LOG_DIR = OUT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("nmdc_lakehouse.taxonomy")
logger.setLevel(logging.INFO)
for _h in list(logger.handlers):
    logger.removeHandler(_h)
_fh = logging.FileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(_sh)
logger.propagate = False

print(f"OUT_DIR:   {OUT_DIR.resolve()}")
print(f"CACHE_DIR: {CACHE_DIR.resolve()}")
print(f"LOG_FILE:  {LOG_FILE.resolve()}")

OUT_DIR:   /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_taxonomy
CACHE_DIR: /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_taxonomy/raw_cache
LOG_FILE:  /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_taxonomy/logs/run_20260501_153845.log


In [2]:
# ── Spark Connect session ─────────────────────────────────────────────────────
# get_spark_session() is the canonical BERDL helper — it wires up Spark Connect auth
# (x-kbase-token header), Hive metastore, Delta, MinIO/S3, and the user's warehouse.

spark = get_spark_session(app_name="fetch_taxonomy_summaries")
print(f"Spark version: {spark.version}")

Spark version: 4.0.1


In [3]:
# ── User configuration ───────────────────────────────────────────────────────
# Edit FETCH_TYPES to limit which data types are fetched this run.
# Set to None (or leave as the full list) to fetch everything.
# The ingest notebook auto-skips types already registered in nmdc_results,
# so over-fetching is safe — just slower.
FETCH_TYPES = None  # fetch all supported types
# FETCH_TYPES = ["Annotation Statistics"]
# FETCH_TYPES = ["Centrifuge output report file"]
# FETCH_TYPES = ["Annotation Statistics", "Centrifuge output report file"]

# ── Supported types (do not edit) ────────────────────────────────────────────
# Validated against nmdc-schema FileTypeEnum at notebook startup, so a
# rename in nmdc-schema fails fast here instead of returning an empty manifest.
from nmdc_lakehouse.file_types import resolve_file_types

_DEFAULT_TARGET_TYPES = resolve_file_types([
    "GOTTCHA2 Classification Report",
    "Kraken2 Classification Report",
    "Centrifuge output report file",
    "GTDBTK Bacterial Summary",
    "GTDBTK Archaeal Summary",
    "CheckM Statistics",
    "Annotation Statistics",
])

# TAXONOMY_TYPES env var takes precedence over FETCH_TYPES (for agent/automated use).
_env = os.environ.get("TAXONOMY_TYPES", "").strip()
if _env:
    TARGET_TYPES = [s.strip() for s in _env.split(",") if s.strip()]
    print(f"TAXONOMY_TYPES env override: {TARGET_TYPES}")
elif FETCH_TYPES is not None:
    TARGET_TYPES = list(FETCH_TYPES)
else:
    TARGET_TYPES = list(_DEFAULT_TARGET_TYPES)

_unknown = [t for t in TARGET_TYPES if t not in _DEFAULT_TARGET_TYPES]
if _unknown:
    raise ValueError(f"Unknown type(s): {_unknown}. Allowed: {_DEFAULT_TARGET_TYPES}")

print(f"Fetching {len(TARGET_TYPES)} type(s): {TARGET_TYPES}")

type_list = ", ".join(f"'{t}'" for t in TARGET_TYPES)

manifest_sql = f"""
SELECT id, url, data_object_type, was_generated_by, file_size_bytes, md5_checksum
FROM nmdc_metadata.data_object_set
WHERE data_object_type IN ({type_list})
  AND url IS NOT NULL
  AND url LIKE 'https://data.microbiomedata.org/%'
ORDER BY data_object_type, id
"""

Fetching 2 type(s): ['Annotation Statistics', 'Centrifuge output report file']


In [4]:
t_start = time.monotonic()
print(f"start:  {datetime.now().isoformat(timespec='seconds')}")

start:  2026-05-01T15:38:46


In [5]:
manifest = spark.sql(manifest_sql).toPandas()

In [6]:
t_end = time.monotonic()
print(f"end:    {datetime.now().isoformat(timespec='seconds')}")
print(f"elapsed: {t_end - t_start:.1f}s")

# The manifest can have duplicate (url, data_object_type) rows — different data_object
# IDs pointing at the same physical file. Dedupe so we don't fetch/parse the same file
# twice and so parallel workers don't race on the same cache file.
n_before = len(manifest)
manifest = manifest.drop_duplicates(subset=["url", "data_object_type"]).reset_index(drop=True)
if len(manifest) != n_before:
    print(f"deduped: {n_before:,} → {len(manifest):,} ({n_before - len(manifest):,} duplicate (url, type) rows dropped)")

print(f"{len(manifest):,} data objects")
print(manifest.groupby("data_object_type").size().to_string())

end:    2026-05-01T15:38:48
elapsed: 1.9s
9,256 data objects
data_object_type
Annotation Statistics            4824
Centrifuge output report file    4432


In [7]:
# ── Format parsers ────────────────────────────────────────────────────────────

# Many NMDC workflow runs produce a single-line placeholder file when there are no
# results (instead of an empty file or no file). Detecting these up front lets parsers
# return None so they're excluded from the merged DataFrame, instead of polluting the
# schema with one bogus column-header per placeholder.
_PLACEHOLDER_PATTERNS = (
    "No Checkm Results for",
    "Nothing found in gottcha2 for",
    "No Archaeal Results for",
    "No Bacterial Results for",
    "Nothing found in kraken2 for",   # speculative, same convention if Kraken2 uses it
    "No Kraken Results for",          # speculative
    "Nothing found in centrifuge for",  # speculative, same convention defensively
    "No Centrifuge Results for",        # speculative
    "Nothing found in annotation for",   # speculative, same convention defensively
    "No Annotation Results for",         # speculative
)


def _is_placeholder(text: str) -> bool:
    stripped = text.strip()
    if not stripped:
        return True
    if "\n" in stripped:
        return False  # multi-line content is real data
    return any(stripped.startswith(p) for p in _PLACEHOLDER_PATTERNS)


# GOTTCHA2: headered TSV
def parse_gottcha2(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    return pd.read_csv(io.StringIO(text), sep="\t")


# Kraken2: no header, 6 cols
_KRAKEN2_COLS = ["pct_clade", "clade_reads", "direct_reads", "rank", "taxid", "name"]
def parse_kraken2(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    return pd.read_csv(io.StringIO(text), sep="\t", header=None, names=_KRAKEN2_COLS)


# Centrifuge: headered TSV
# columns: name, taxID, taxRank, genomeSize, numReads, numUniqueReads, abundance
def parse_centrifuge(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    return pd.read_csv(io.StringIO(text), sep="\t")


# GTDB-tk: headered TSV
def parse_gtdbtk(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    df = pd.read_csv(io.StringIO(text), sep="\t")
    if "user_genome" in df.columns and "name" not in df.columns:
        df = df.rename(columns={"user_genome": "name"})
    return df


# CheckM qa output: multi-space-delimited table interleaved with `---` separator lines.
# 14 columns: bin_id, marker_lineage, # genomes, # markers, # marker sets,
#             0, 1, 2, 3, 4, 5+, Completeness, Contamination, Strain heterogeneity.
_CHECKM_COLS = [
    "bin_id", "marker_lineage", "n_genomes", "n_markers", "n_marker_sets",
    "n_0", "n_1", "n_2", "n_3", "n_4", "n_5_plus",
    "completeness", "contamination", "strain_heterogeneity",
]
_CHECKM_NUMERIC_COLS = _CHECKM_COLS[2:]
_CHECKM_SEP_RE = re.compile(r"^\s*-{3,}\s*$")


def parse_checkm(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    rows: list[list[str]] = []
    for line in text.splitlines():
        if not line.strip() or _CHECKM_SEP_RE.match(line):
            continue
        fields = re.split(r"\s{2,}", line.strip())
        if len(fields) != len(_CHECKM_COLS):
            continue
        if fields[0] == "Bin Id":
            continue  # literal header row (data rows look like "bins.1")
        rows.append(fields)
    if not rows:
        return None
    df = pd.DataFrame(rows, columns=_CHECKM_COLS)
    for col in _CHECKM_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


# Annotation Statistics: tab-separated multi-section report with three sections:
#   1. Processed Sequences Statistics (3 rows: final_fasta, sequences_with_genes,
#      sequences_without_genes)
#   2. Predicted Genes Statistics (variable rows, one per feature_type/method pair)
#   3. General Quality Info (1 row)
# Sections are delimited by lines of === (10+) and --- (10+).
# Output: one wide row per workflow run with key QC metrics from all three sections.

def _annotation_stats_sections(text: str) -> dict:
    """Split the multi-section TSV into {section_title: DataFrame}."""
    sections = {}
    lines = text.splitlines()
    i = 0
    while i < len(lines):
        stripped = lines[i].strip()
        if not stripped or re.match(r'^[=\-]{10,}', stripped):
            i += 1
            continue
        j = i + 1
        while j < len(lines) and not lines[j].strip():
            j += 1
        if j < len(lines) and re.match(r'^={10,}', lines[j].strip()):
            title = stripped
            i = j + 1
            header = None
            rows = []
            while i < len(lines):
                l = lines[i].strip()
                if re.match(r'^={10,}', l):
                    i += 1
                    break
                if re.match(r'^-{10,}', l) or not l:
                    i += 1
                    continue
                if header is None:
                    header = lines[i].split("\t")
                else:
                    rows.append(lines[i].split("\t"))
                i += 1
            if header and rows:
                sections[title] = pd.DataFrame(rows, columns=header)
        else:
            i += 1
    return sections


def parse_annotation_statistics(text: str) -> pd.DataFrame | None:
    if _is_placeholder(text):
        return None
    secs = _annotation_stats_sections(text)
    if not secs:
        return None
    out = {}

    seq = secs.get("Processed Sequences Statistics")
    if seq is not None and not seq.empty:
        seq["Data type"] = seq["Data type"].str.strip("'")
        seq = seq.set_index("Data type")
        def _si(row, col):
            try: return int(seq.at[row, col])
            except (KeyError, ValueError): return None
        def _sf(row, col):
            try: return float(seq.at[row, col])
            except (KeyError, ValueError): return None
        out["n_seqs"]               = _si("final_fasta", "Number of seqs")
        out["n_bps"]                = _si("final_fasta", "Number of bps")
        out["len_shortest_seq"]     = _si("final_fasta", "Length shortest seq")
        out["len_longest_seq"]      = _si("final_fasta", "Length longest seq")
        out["avg_seq_length"]       = _sf("final_fasta", "Average length")
        out["median_seq_length"]    = _sf("final_fasta", "Median length")
        out["stddev_seq_length"]    = _sf("final_fasta", "Standard deviation")
        out["n_seqs_with_genes"]    = _si("sequences_with_genes", "Number of seqs")
        out["n_seqs_without_genes"] = _si("sequences_without_genes", "Number of seqs")

    gene = secs.get("Predicted Genes Statistics")
    if gene is not None and not gene.empty:
        gene["Feature type"] = gene["Feature type"].str.strip("'")
        n_feat = (
            gene.assign(_n=pd.to_numeric(gene["Number of predicted features"], errors="coerce"))
            .groupby("Feature type")["_n"].sum()
        )
        for feat, col in [("CDS", "n_cds"), ("tRNA", "n_trna"), ("ncRNA", "n_ncrna"),
                          ("rRNA", "n_rrna"), ("CRISPR", "n_crispr")]:
            out[col] = int(n_feat[feat]) if feat in n_feat.index else pd.NA

    qual = secs.get("General Quality Info")
    if qual is not None and not qual.empty:
        r = qual.iloc[0]
        try:
            out["coding_density_pct"] = float(str(r.get("Coding density", "")).rstrip("%"))
        except ValueError:
            out["coding_density_pct"] = None
        try:
            out["genes_per_1m_bp"] = float(r.get("Genes per 1M bp", None))
        except (ValueError, TypeError):
            out["genes_per_1m_bp"] = None
        try:
            out["seqs_per_1m_bp"] = float(r.get("Seqs per 1M bp", None))
        except (ValueError, TypeError):
            out["seqs_per_1m_bp"] = None

    return pd.DataFrame([out]) if out else None


_PARSERS = {
    "GOTTCHA2 Classification Report": parse_gottcha2,
    "Kraken2 Classification Report": parse_kraken2,
    "Centrifuge output report file": parse_centrifuge,
    "GTDBTK Bacterial Summary": parse_gtdbtk,
    "GTDBTK Archaeal Summary": parse_gtdbtk,
    "CheckM Statistics": parse_checkm,
    "Annotation Statistics": parse_annotation_statistics,
}

In [8]:
# ── Download + parse loop (parallel, with disk cache) ────────────────────────

import threading

_thread_local = threading.local()


def _get_session() -> requests.Session:
    if not hasattr(_thread_local, "session"):
        _thread_local.session = requests.Session()
        _thread_local.session.headers.update({"User-Agent": "nmdc-lakehouse/fetch_taxonomy_summaries"})
    return _thread_local.session


@retry(
    retry=retry_if_exception_type(requests.RequestException),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    reraise=True,
)
def fetch_text(url: str, timeout: int = 60) -> str:
    r = _get_session().get(url, timeout=timeout)
    r.raise_for_status()
    return r.text


def _cache_path_for(url: str) -> Path:
    return CACHE_DIR / urlparse(url).path.lstrip("/")


def fetch_text_cached(url: str, timeout: int = 60) -> str:
    """Return file contents, fetching from network only if not already cached on disk."""
    path = _cache_path_for(url)
    if path.exists():
        return path.read_text()
    text = fetch_text(url, timeout=timeout)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Per-thread temp suffix avoids the race where two threads fetch the same URL
    # (manifest can contain duplicates) and one's atomic rename consumes the other's tmp.
    tmp = path.with_suffix(f"{path.suffix}.{threading.get_ident()}.tmp")
    tmp.write_text(text)
    tmp.replace(path)
    return text


def _fetch_and_parse(row, parser):
    """Returns (df_or_None, error_or_None). df=None & error=None means 'placeholder file'."""
    try:
        text = fetch_text_cached(row.url)
        df = parser(text)
        if df is None:
            return None, None  # placeholder / no-results — not an error
        df["workflow_run_id"] = row.was_generated_by
        df["data_object_id"] = row.id
        return df, None
    except Exception as exc:
        return None, (row.url, str(exc))


def _stabilize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Stabilize column dtypes after concat across heterogeneously-typed per-file frames.

    Per-file pandas type inference can disagree across files (e.g. GOTTCHA2's TAXID is
    int in most files but str in a few like '2759_pt'). After concat the column has
    dtype=object with mixed Python types, which pyarrow refuses to write.

    Strategy: for each object column, try numeric coercion. If every non-null value
    parses as numeric, keep it numeric. Otherwise fall back to nullable string.
    """
    for col in df.columns:
        if df[col].dtype != object:
            continue
        try:
            # errors='raise' fails fast on the first non-numeric value
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            df[col] = df[col].astype("string")
    return df


def load_type(type_name: str, rows: pd.DataFrame) -> pd.DataFrame | None:
    parser = _PARSERS[type_name]
    frames: list[pd.DataFrame] = []
    errors: list[tuple[str, str]] = []
    n_placeholders = 0
    n_total_in_manifest = len(rows)
    if DRY_RUN:
        rows = rows.head(1)
        logger.info(f"  DRY_RUN: processing 1 of {n_total_in_manifest} files")

    total = len(rows)
    cache_hits = sum(1 for r in rows.itertuples() if _cache_path_for(r.url).exists())
    logger.info(f"  cache: {cache_hits:,}/{total:,} already on disk; will fetch {total - cache_hits:,}")

    with ThreadPoolExecutor(max_workers=HTTP_WORKERS) as ex:
        futures = [ex.submit(_fetch_and_parse, row, parser) for row in rows.itertuples()]
        for i, fut in enumerate(as_completed(futures), 1):
            df, err = fut.result()
            if df is not None:
                frames.append(df)
            elif err is not None:
                errors.append(err)
            else:
                n_placeholders += 1
            if i % 500 == 0 or i == 1 or i == total:
                print(f"  {i}/{total}…", end="", flush=True)

    print()  # newline after progress
    logger.info(
        f"  parsed: {len(frames):,} files with data; "
        f"{n_placeholders:,} placeholders skipped; {len(errors):,} errors"
    )
    if errors:
        for url, msg in errors[:5]:
            logger.info(f"    error: {url}: {msg}")
        if len(errors) > 5:
            logger.info(f"    … and {len(errors) - 5} more (full log file has all of them)")

    if not frames:
        if errors:
            logger.info("  skipped (all errors — no data written)")
        else:
            logger.info("  skipped (all placeholders — no data for this type)")
        return None
    combined = pd.concat(frames, ignore_index=True)
    combined = _stabilize_dtypes(combined)
    logger.info(f"  combined: {len(combined):,} rows × {len(combined.columns)} cols")
    logger.info(f"  dtypes: {dict(combined.dtypes.astype(str))}")
    return combined

In [9]:
# ── Main: one Parquet per data_object_type ────────────────────────────────────

# Sanity check: fail fast if the kernel has stale code from a previous edit
assert 'fields[0] == "Bin Id"' in inspect.getsource(parse_checkm), \
    "Kernel has STALE parse_checkm — re-run the parsers cell."
assert "_stabilize_dtypes" in inspect.getsource(load_type), \
    "Kernel has STALE load_type — re-run the load_type cell."
assert "fetch_text_cached" in inspect.getsource(_fetch_and_parse), \
    "Kernel has STALE _fetch_and_parse — re-run the load_type cell."
assert "_annotation_stats_sections" in inspect.getsource(parse_annotation_statistics), \
    "Kernel has STALE parse_annotation_statistics — re-run the parsers cell."
logger.info("Sanity check passed: in-kernel parsers and loader are current.")


def _safe(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


summary: dict[str, int] = {}
t0 = time.monotonic()

for type_name, group in manifest.groupby("data_object_type"):
    out_path = OUT_DIR / f"{_safe(type_name)}.parquet"
    logger.info(f"\n{'─'*60}")
    logger.info(f"{type_name} ({len(group):,} files → {out_path.name})")

    combined = load_type(type_name, group)
    if combined is None:
        continue

    combined.to_parquet(out_path, index=False)
    summary[type_name] = len(combined)
    logger.info(f"  wrote {out_path}")

elapsed = time.monotonic() - t0
logger.info(f"\n{'='*60}")
logger.info(f"Done in {elapsed/60:.1f} min")
for t, n in summary.items():
    logger.info(f"  {t}: {n:,} rows")
logger.info(f"Full log: {LOG_FILE}")

Sanity check passed: in-kernel parsers and loader are current.

────────────────────────────────────────────────────────────
Annotation Statistics (4,824 files → annotation_statistics.parquet)
  cache: 0/4,824 already on disk; will fetch 4,824


  1/4824…  500/4824…  1000/4824…  1500/4824…  2000/4824…  2500/4824…  3000/4824…  3500/4824…  4000/4824…  4500/4824…  4824/4824…

  parsed: 4,815 files with data; 0 placeholders skipped; 9 errors
    error: https://data.microbiomedata.org/data/nmdc:omprc-11-x7sz0s85/nmdc:wfmgan-11-7ns8za11.1/nmdc_wfmgan-11-7ns8za11.1_stats.tsv: 404 Client Error: Not Found for url: https://data.microbiomedata.org/data/nmdc:omprc-11-x7sz0s85/nmdc:wfmgan-11-7ns8za11.1/nmdc_wfmgan-11-7ns8za11.1_stats.tsv
    error: https://data.microbiomedata.org/data/nmdc:omprc-11-kr53zx90/nmdc:wfmgan-11-20jhbp19.1/nmdc_wfmgan-11-20jhbp19.1_stats.tsv: 404 Client Error: Not Found for url: https://data.microbiomedata.org/data/nmdc:omprc-11-kr53zx90/nmdc:wfmgan-11-20jhbp19.1/nmdc_wfmgan-11-20jhbp19.1_stats.tsv
    error: https://data.microbiomedata.org/data/nmdc:omprc-11-5ghk8986/nmdc:wfmgan-11-64we1x93.1/nmdc_wfmgan-11-64we1x93.1_stats.tsv: 404 Client Error: Not Found for url: https://data.microbiomedata.org/data/nmdc:omprc-11-5ghk8986/nmdc:wfmgan-11-64we1x93.1/nmdc_wfmgan-11-64we1x93.1_stats.tsv
    error: https://data.microbiomedata.org/data/nmdc:om

  combined: 4,815 rows × 19 cols
  dtypes: {'n_seqs': 'int64', 'n_bps': 'int64', 'len_shortest_seq': 'int64', 'len_longest_seq': 'int64', 'avg_seq_length': 'float64', 'median_seq_length': 'float64', 'stddev_seq_length': 'float64', 'n_seqs_with_genes': 'int64', 'n_seqs_without_genes': 'float64', 'n_cds': 'int64', 'n_trna': 'float64', 'n_ncrna': 'float64', 'n_rrna': 'float64', 'n_crispr': 'float64', 'coding_density_pct': 'float64', 'genes_per_1m_bp': 'float64', 'seqs_per_1m_bp': 'float64', 'workflow_run_id': 'string', 'data_object_id': 'str'}
  wrote loaded_taxonomy/annotation_statistics.parquet

────────────────────────────────────────────────────────────
Centrifuge output report file (4,432 files → centrifuge_output_report_file.parquet)
  cache: 0/4,432 already on disk; will fetch 4,432


  1/4432…  500/4432…  1000/4432…  1500/4432…  2000/4432…  2500/4432…  3000/4432…  3500/4432…  4000/4432…  4432/4432…

  parsed: 4,423 files with data; 1 placeholders skipped; 8 errors
    error: https://data.microbiomedata.org/data/nmdc:omprc-11-92rmyt64/nmdc:wfrbt-11-92skkb23.1/nmdc_wfrbt-11-92skkb23.1_centrifuge_report.tsv: 404 Client Error: Not Found for url: https://data.microbiomedata.org/data/nmdc:omprc-11-92rmyt64/nmdc:wfrbt-11-92skkb23.1/nmdc_wfrbt-11-92skkb23.1_centrifuge_report.tsv
    error: https://data.microbiomedata.org/data/nmdc:omprc-11-dkrhkz48/nmdc:wfrbt-11-01v9yh59.1/nmdc_wfrbt-11-01v9yh59.1_centrifuge_report.tsv: 404 Client Error: Not Found for url: https://data.microbiomedata.org/data/nmdc:omprc-11-dkrhkz48/nmdc:wfrbt-11-01v9yh59.1/nmdc_wfrbt-11-01v9yh59.1_centrifuge_report.tsv
    error: https://data.microbiomedata.org/data/nmdc:omprc-11-yrybv196/nmdc:wfrbt-11-w8scgv45.1/nmdc_wfrbt-11-w8scgv45.1_centrifuge_report.tsv: 404 Client Error: Not Found for url: https://data.microbiomedata.org/data/nmdc:omprc-11-yrybv196/nmdc:wfrbt-11-w8scgv45.1/nmdc_wfrbt-11-w8scgv45.1_centrifuge_report

  combined: 20,564,570 rows × 9 cols
  dtypes: {'name': 'str', 'taxID': 'int64', 'taxRank': 'str', 'genomeSize': 'int64', 'numReads': 'int64', 'numUniqueReads': 'int64', 'abundance': 'float64', 'workflow_run_id': 'string', 'data_object_id': 'str'}
  wrote loaded_taxonomy/centrifuge_output_report_file.parquet

Done in 4.7 min
  Annotation Statistics: 4,815 rows
  Centrifuge output report file: 20,564,570 rows
Full log: loaded_taxonomy/logs/run_20260501_153845.log


In [10]:
# ── Spot-check: GOTTCHA2 ──────────────────────────────────────────────────────

import pyarrow.parquet as pq

g2_path = OUT_DIR / "gottcha2_classification_report.parquet"
if g2_path.exists():
    g2 = pd.read_parquet(g2_path)
    print(g2.shape)
    print(g2.dtypes)
    print(g2.head(3).to_string())
    print(f"\nDistinct workflow runs: {g2['workflow_run_id'].nunique():,}")
    print(f"Distinct taxa (NAME): {g2['NAME'].nunique():,}")

(673049, 12)
LEVEL                 string
NAME                  string
TAXID                 string
READ_COUNT             int64
TOTAL_BP_MAPPED        int64
TOTAL_BP_MISMATCH      int64
LINEAR_LEN             int64
LINEAR_DOC           float64
ROLLUP_DOC           float64
REL_ABUNDANCE        float64
workflow_run_id       string
data_object_id           str
dtype: object
          LEVEL            NAME   TAXID  READ_COUNT  TOTAL_BP_MAPPED  TOTAL_BP_MISMATCH  LINEAR_LEN  LINEAR_DOC  ROLLUP_DOC  REL_ABUNDANCE           workflow_run_id         data_object_id
0  superkingdom        Bacteria       2         671            61559               2053       38310    1.606865    0.034644       1.000000  nmdc:wfrbt-11-y6rnfz23.1  nmdc:dobj-11-ar1e9819
1        phylum  Pseudomonadota    1224         658            60381               2024       37587    1.606433    0.020266       0.584994  nmdc:wfrbt-11-y6rnfz23.1  nmdc:dobj-11-ar1e9819
2        phylum   Chloroflexota  200795          13          

## Next steps

1. **Upload to BERDL object storage** — use the BERDL upload utilities or `aws s3 cp` to push the Parquet files to the nmdc lakehouse bucket.
2. **Add biosample join** — join `workflow_run_id` to `nmdc_metadata.metagenome_annotation_activity_set.was_informed_by` → omics processing run → `has_input` → biosample ID.
3. **Register tables in BERDL catalog** — so the files are queryable via `nmdc_metadata.<type>` SQL.
4. **Extend to LATER tier** (issue #78) — KO/EC TSVs and Functional Annotation GFF require the same HTTP approach but are much larger (163+111 GB); streaming/chunked writes will be needed.